In [ ]:
from unet import UNetAutoencode
from quantizedClassifier import QuantizedClassifier
from dataset import SemiSupervisedMnistDataModule

from hydra import initialize, compose
from omegaconf import OmegaConf
import random, numpy as np

In [2]:
# Ajusta config_path si tu carpeta conf está en otra ruta
with initialize(config_path="../config", version_base=None):
    cfg = compose(
        config_name="config",
        # aquí puedes sobrescribir parámetros si quieres
        overrides=[
            # e.g.: "batch_size=32", "optimizer.name=SGD", "learning_rate=0.01"
        ]
    )

# Opcional: imprime la config para verificar
print(OmegaConf.to_yaml(cfg))

# Fija semillas
random.seed(cfg.seed)
np.random.seed(cfg.seed)

optimizer:
  _name_: adam
  optimizer:
    name: Adam
    weight_decay: 0.0001
seed: 42
batch_size: 64
num_workers: 4
label_pct: 0.3
max_epochs: 20
early_stop_patience: 3
learning_rate: 0.0005
project_name: autoencoder-clasificador
data_dir: ../data



In [3]:
# Configuraciones básicas
MAX_EPOCHS          = cfg.max_epochs
EARLY_STOP_PATIENCE = cfg.early_stop_patience
PROJECT_NAME        = cfg.project_name

## Experimento 70% sin labels y 30% con labels

In [4]:
dm_70 = SemiSupervisedMnistDataModule(
    data_dir="../data/",
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    label_pct=0.3 
)
dm_70.setup()


In [5]:

import pytorch_lightning as pl
import torch

def test(model, datamodule):
    correct = 0
    total = 0

    iterations = 0

    # Force device to CPU for quantized models
    if hasattr(model, 'is_quantized') or 'Quantized' in model.__class__.__name__:
        device = torch.device("cpu")
        model.cpu()
    else:
        try:
            device = next(model.parameters()).device
        except StopIteration:
            device = torch.device("cpu")
    model.eval()

    with torch.no_grad():
        for data in datamodule.test_dataloader():
            x, y = data
            x = x.to(device)
            y = y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()
            iterations += 1
            if iterations >= 100:
                break
    accuracy = 100 * correct / total

    print(f'Accuracy of the model on the test images: {accuracy:.2f}%')

    

### Cargar Checkpoint 

In [6]:
pretrained_autoencoder_70 = UNetAutoencoder.load_from_checkpoint("../output/checkpoints/unet/70/epoch=18-val_loss=0.00022231.ckpt")
pretrained_autoencoder_90 = UNetAutoencoder.load_from_checkpoint("../output/checkpoints/unet/90/epoch=18-val_loss=0.00016913.ckpt")
raw_autoencoder = UNetAutoencoder()

## Quatization 

Mejores modelos:
- B2_70 (1.9)
- A_70 (2.2)
- B1_70 (2.8)

In [7]:
import torch
import os

In [8]:
def print_size_of_model(model):
    torch.save(model.state_dict(), "temp_delme.p")
    print('Size (KB):', os.path.getsize("temp_delme.p")/1e3)
    os.remove('temp_delme.p')

In [9]:
# load the best checkpoints
quantized_b2_70 = QuantizedClassifier.load_from_checkpoint("../output/checkpoints/classifierB2/70/epoch=19-val_loss=1.81377745.ckpt", encoder=raw_autoencoder.get_encoder())

quantized_a_70 = QuantizedClassifier.load_from_checkpoint("../output/checkpoints/classifierA/70/epoch=19-val_loss=2.33019876.ckpt", encoder=pretrained_autoencoder_70.get_encoder())

quantized_b1_70 = QuantizedClassifier.load_from_checkpoint("../output/checkpoints/classifierB1/70/epoch=19-val_loss=2.80386853.ckpt", encoder=pretrained_autoencoder_70.get_encoder())

In [10]:
print("Before Quantization B2 70:")
test(quantized_b2_70, dm_70)
print_size_of_model(quantized_b2_70)

print("\nBefore Quantization A 70:")
test(quantized_a_70, dm_70)
print_size_of_model(quantized_a_70)

print("\nBefore Quantization B1 70:")
test(quantized_b1_70, dm_70)
print_size_of_model(quantized_b1_70)

Before Quantization B2 70:
Accuracy of the model on the test images: 44.00%
Size (KB): 9473.442
Before Quantization A 70:
Accuracy of the model on the test images: 24.67%
Size (KB): 9473.442
Before Quantization B1 70:
Accuracy of the model on the test images: 16.80%
Size (KB): 9473.442


In [11]:
quantized_b2_70.eval()

quantized_b2_70.qconfig = torch.ao.quantization.default_qconfig # Which layers will be quantized
quantized_b2_70 = torch.ao.quantization.prepare(quantized_b2_70) # Insert observers
quantized_b2_70

quantized_a_70.qconfig = torch.ao.quantization.default_qconfig # Which layers will be quantized
quantized_a_70 = torch.ao.quantization.prepare(quantized_a_70) # Insert observers
quantized_a_70

quantized_b1_70.qconfig = torch.ao.quantization.default_qconfig # Which layers will be quantized
quantized_b1_70 = torch.ao.quantization.prepare(quantized_b1_70) # Insert observers
quantized_b1_70

QuantizedClassifier(
  (encoder): Sequential(
    (0): DoubleConv(
      (double_conv): Sequential(
        (0): Conv2d(
          3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
          (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
        )
        (1): ReLU(inplace=True)
        (2): Conv2d(
          64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
          (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
        )
        (3): ReLU(inplace=True)
      )
    )
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): DoubleConv(
      (double_conv): Sequential(
        (0): Conv2d(
          64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
          (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
        )
        (1): ReLU(inplace=True)
        (2): Conv2d(
          128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
          (activation_post_

In [12]:
test(quantized_b2_70, dm_70)
test(quantized_a_70, dm_70)
test(quantized_b1_70, dm_70)


Accuracy of the model on the test images: 44.00%
Accuracy of the model on the test images: 24.67%
Accuracy of the model on the test images: 16.80%


In [13]:
quantized_b2_70 = torch.ao.quantization.convert(quantized_b2_70)
quantized_a_70 = torch.ao.quantization.convert(quantized_a_70)
quantized_b1_70 = torch.ao.quantization.convert(quantized_b1_70)

In [14]:
print('Testing the models after quantization')

print("After Quantization B2 70:")
test(quantized_b2_70, dm_70)
print_size_of_model(quantized_b2_70)

print("\nAfter Quantization A 70:")
test(quantized_a_70, dm_70)
print_size_of_model(quantized_a_70)

print("\nAfter Quantization B1 70:")
test(quantized_b1_70, dm_70)
print_size_of_model(quantized_b1_70)

Testing the models after quantization
After Quantization B2 70:
Accuracy of the model on the test images: 42.53%
Size (KB): 2386.914

After Quantization A 70:
Accuracy of the model on the test images: 25.87%
Size (KB): 2386.914

After Quantization B1 70:
Accuracy of the model on the test images: 14.80%
Size (KB): 2386.914
